<a href="https://colab.research.google.com/github/Ruba-Alharbi/MentalQA_NER-Reproduction/blob/main/RP_Zero_shot_LLM_as_a_judge.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

##  Import Libraries & Load the data

In [ ]:
!pip install openai tqdm

In [ ]:
import pandas as pd
import openai
import json
from tqdm import tqdm
import os

In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=False)

Mounted at /content/drive


# Data Cleaning

In [ ]:
import json
import re
from ast import literal_eval

import pandas as pd
from openpyxl import load_workbook
from openpyxl.styles import Font

# Load the file containing corrected GPT-4o and LLaMA zero-shot outputs.
gpt_llama_df = pd.read_excel("llama_df_zeroshot_retried.xlsx")

# Load the ALLaM zero-shot outputs.
allam_df = pd.read_excel("zero-allam_df.xlsx")

# Keep only the required columns.
gpt_llama_df = gpt_llama_df[
    ["question", "GPT-4o Output", "LLaMA Output"]
]

allam_df = allam_df[
    ["question", "Allam Output"]
]

# Merge ALLaM outputs with GPT-4o and corrected LLaMA outputs.
zero_all = gpt_llama_df.merge(
    allam_df,
    on="question",
    how="left",
    validate="one_to_one"
)

# Validate that all raw outputs are present before cleaning.
output_columns = [
    "GPT-4o Output",
    "LLaMA Output",
    "Allam Output"
]

missing_outputs = zero_all[output_columns].isna().sum()

if missing_outputs.any():
    raise RuntimeError(
        f"Missing raw outputs detected:\n{missing_outputs}"
    )

llama_api_errors = zero_all["LLaMA Output"].fillna("").astype(str).str.startswith(
    "LLaMA Error:"
).sum()

if llama_api_errors > 0:
    raise RuntimeError(
        f"Remaining LLaMA API errors: {llama_api_errors}"
    )

# Define the entity schema used by the published cleaning function.
expected_keys = [
    "Symptom", "Diagnosis", "Treatment", "Drug", "Dosage",
    "Duration", "Strength", "Unit", "Risk"
]

def clean_model_output(text):
    """Clean GPT-4o or LLaMA output into normalized JSON."""
    try:
        if pd.isna(text) or not isinstance(text, str) or text.strip() == "":
            return json.dumps(
                [{key: "" for key in expected_keys}],
                ensure_ascii=False
            )

        text = text.strip()
        text = re.sub(r"^.*?\[", "[", text, flags=re.DOTALL)

        match = re.search(r"\[.*\]", text, re.DOTALL)
        if not match:
            return json.dumps(
                [{key: "" for key in expected_keys}],
                ensure_ascii=False
            )

        json_string = match.group(0)

        try:
            parsed = literal_eval(json_string)
        except Exception:
            parsed = json.loads(json_string)

        if isinstance(parsed, list):
            cleaned = []

            for entry in parsed:
                if isinstance(entry, dict):
                    cleaned.append({
                        key: str(entry.get(key, "")).strip()
                        for key in expected_keys
                    })

            return json.dumps(cleaned, ensure_ascii=False)

        return json.dumps(
            [{key: "" for key in expected_keys}],
            ensure_ascii=False
        )

    except Exception:
        return json.dumps(
            [{key: "" for key in expected_keys}],
            ensure_ascii=False
        )


def clean_allam_output(text):
    """Clean ALLaM output using the published notebook procedure."""
    text = str(text).strip()

    text = re.sub(
        r"^(?:الإخراج|Output)\s*:\s*",
        "",
        text,
        flags=re.IGNORECASE
    )

    if not text:
        return "no answer"

    match = re.search(r"\[\s*\{.*?\}\s*\]", text, flags=re.DOTALL)
    if match:
        return match.group(0)

    start_index = text.rfind("[{")

    if start_index != -1:
        candidate = text[start_index:].strip()
        closing_match = re.search(r"\}\s*\]", candidate, flags=re.DOTALL)

        if closing_match:
            return candidate[:closing_match.end()]

        return candidate + "}]"

    return "no answer"


# Apply the same cleaning logic used before LLM-as-a-Judge evaluation.
zero_all["GPT-4o Cleaned"] = zero_all["GPT-4o Output"].apply(
    clean_model_output
)

zero_all["LLaMA Cleaned"] = zero_all["LLaMA Output"].apply(
    clean_model_output
)

zero_all["ALLaM Cleaned"] = zero_all["Allam Output"].apply(
    clean_allam_output
)

# Save the corrected zero-shot Judge input.
output_file = "zero_all_cleaned_retried.xlsx"
zero_all.to_excel(output_file, index=False)

# Apply basic formatting.
workbook = load_workbook(output_file)
worksheet = workbook.active
worksheet.freeze_panes = "A2"
worksheet.auto_filter.ref = worksheet.dimensions

for row in worksheet.iter_rows():
    for cell in row:
        cell.font = Font(name="Arial", size=10)

for cell in worksheet[1]:
    cell.font = Font(name="Arial", size=10, bold=True)

workbook.save(output_file)

print(f"Rows: {len(zero_all)}")
print("Missing raw outputs:")
print(missing_outputs)

print(f"Remaining LLaMA API errors: {llama_api_errors}")
print(f"Saved: {output_file}")

print("\nRequired Judge columns:")
print(
    zero_all[
        [
            "question",
            "GPT-4o Cleaned",
            "LLaMA Cleaned",
            "ALLaM Cleaned"
        ]
    ].columns.tolist()
)

Rows: 499
Missing raw outputs:
GPT-4o Output    0
LLaMA Output     0
Allam Output     0
dtype: int64
Remaining LLaMA API errors: 0
Saved: zero_all_cleaned_retried.xlsx

Required Judge columns:
['question', 'GPT-4o Cleaned', 'LLaMA Cleaned', 'ALLaM Cleaned']


In [ ]:
zero_all = pd.read_excel("zero_all_cleaned_retried.xlsx")

In [ ]:
len(zero_all)

499

In [ ]:
zero_all.tail()

,question,GPT-4o Output,LLaMA Output,Allam Output,GPT-4o Cleaned,LLaMA Cleaned,ALLaM Cleaned
494,قلق وتوتر ووسواس قهري وتأتأة في الكلام وعجز دا...,"[{""Symptom"": ""قلق"", ""Diagnosis"": ""وسواس قهري"",...","[{""Symptom"": ""قلق، توتر، وسواس قهري، تأتأة في ...","**الإجابة**:\n\n[{""Symptom"": ""قلق"", ""Diagnosis...","[{""Symptom"": ""قلق"", ""Diagnosis"": ""وسواس قهري"",...","[{""Symptom"": ""قلق، توتر، وسواس قهري، تأتأة في ...","[{""Symptom"": ""قلق"", ""Diagnosis"": ""وسواس قهري"",..."
495,اعاني من الخوف وتنبؤات من كلام اهلي وبصمتهم ان...,"[{""Symptom"": ""الخوف"", ""Diagnosis"": """", ""Treatm...","[{""Symptom"": ""الخوف، تنبؤات من كلام الأهل، بصم...","الإجابة:\n\n[{""Diagnosis"": ""اضطراب القلق"", ""Ri...","[{""Symptom"": ""الخوف"", ""Diagnosis"": """", ""Treatm...","[{""Symptom"": ""الخوف، تنبؤات من كلام الأهل، بصم...","[{""Diagnosis"": ""اضطراب القلق"", ""Risk"": ""الانتح..."
496,لدي خمس سنوات علاج من الذهانية ولم اشفى تماما ...,"[{""Symptom"": """", ""Diagnosis"": ""الذهانية"", ""Tre...","[{""Symptom"": """", ""Diagnosis"": ""الذهانية"", ""Tre...","**الإجابة**:\n\n[{""Dosage"": ""..."", ""Duration"":...","[{""Symptom"": """", ""Diagnosis"": ""الذهانية"", ""Tre...","[{""Symptom"": """", ""Diagnosis"": ""الذهانية"", ""Tre...","[{""Dosage"": ""..."", ""Duration"": ""لم اشف تماما ب..."
497,أحس بفقدان التركيز كأنني أنسى معلومة أخبرتني ب...,"[{""Symptom"": ""فقدان التركيز"", ""Diagnosis"": """",...","[{""Symptom"": ""فقدان التركيز، النوم، الضغط، الغ...","**الإجابة**:\n[{""Symptom"": ""فقدان التركيز"", ""D...","[{""Symptom"": ""فقدان التركيز"", ""Diagnosis"": """",...","[{""Symptom"": ""فقدان التركيز، النوم، الضغط، الغ...","[{""Symptom"": ""فقدان التركيز"", ""Drug"": ""..."", ""..."
498,انا احس فيني الاكتئاب الجزئي او عسر المزاج نفس...,"[{""Symptom"": ""ماعندي شهيه"", ""Diagnosis"": ""الاك...","[{""Symptom"": ""الاكتئاب الجزئي، عسر المزاج، فقد...","الإجابة:\n\n[{""Diagnosis"": ""اكتئاب جزئي"", ""Tre...","[{""Symptom"": ""ماعندي شهيه"", ""Diagnosis"": ""الاك...","[{""Symptom"": """", ""Diagnosis"": """", ""Treatment"":...","[{""Diagnosis"": ""اكتئاب جزئي"", ""Treatment"": ""ال..."


In [ ]:
zero_all.columns

Index(['question', 'GPT-4o Output', 'LLaMA Output', 'Allam Output',
       'GPT-4o Cleaned', 'LLaMA Cleaned', 'ALLaM Cleaned'],
      dtype='object')

In [ ]:
df = zero_all[['question', 'LLaMA Cleaned', 'ALLaM Cleaned', 'GPT-4o Cleaned']]

In [ ]:
df.head()

,question,LLaMA Cleaned,ALLaM Cleaned,GPT-4o Cleaned
0,هل يعتبر الخوف من عدم الإنجاب مستقبلاً حالة عا...,"[{""Symptom"": ""الخوف من عدم الإنجاب"", ""Diagnosi...","[{""Symptom"": ""الخوف من عدم الإنجاب مستقبلاً"", ...","[{""Symptom"": ""الخوف من عدم الإنجاب مستقبلاً"", ..."
1,من سنه تقريبا و انا أذي نفسي ب اكثر من طريقة و...,"[{""Symptom"": ""أذية النفس، أفكار بإنهاء الحياة""...","[{""symptom"": "","", ""diagnosis"": ""من سنه تقريبا ...","[{""Symptom"": """", ""Diagnosis"": """", ""Treatment"":..."
2,السلام عليكم مشكلتي تقتصر على تكرار كلمة معينة...,"[{""Symptom"": ""تكرار كلمة معينة لمدة طويلة من ا...","[{""Symptom"": ""تكرار كلمة معينة لمدة طويلة"", ""R...","[{""Symptom"": ""تكرار كلمة معينة"", ""Diagnosis"": ..."
3,اكتئاب وفوبيا من المجتمع وانعزال وانطوائية وتع...,"[{""Symptom"": ""اكتئاب، فوبيا، انعزال، انطوائية،...","[{""Symptom"": ""اكتئاب"", ""Diagnosis"": ""فوبيا من ...","[{""Symptom"": ""فوبيا من المجتمع، انعزال، انطوائ..."
4,عمري 37 سنة وحامل في الشهر الثامن . منذ الشهر ...,"[{""Symptom"": ""قلق شديد مزمن"", ""Diagnosis"": ""ال...","[{""Symptom"": ""قلق شديد مزمن وخوف كبير"", \n""Dia...","[{""Symptom"": ""قلق شديد مزمن"", ""Diagnosis"": ""ال..."


## Define the Judge Prompt

### Latest Prompt

In [ ]:
def build_prompt(question_text, ner_output):
    return f"""
    You are an expert medical evaluator. Your task is to act as an impartial judge and evaluate the quality of named entity extraction performed by a single AI model. We will provide you with a medical question,
    a model-generated list of extracted entities, and a set of evaluation criteria. You should first read the question carefully, then evaluate the extracted entities based on the following two criteria: Correctness and Completeness.

    For each criterion:
    - Use only the question text as the source of truth.
    - Refer strictly to the entity definitions provided.
    - Do not infer or hallucinate missing information.
    - Output your final verdict using one of the valid options for each criterion (see below).
    - Provide a short explanation for your decision.

    ## Question:
    {question_text}

    ## Model Output (NER Prediction):
    {ner_output}

    ## Entity Schema:
    Evaluate only the following entity types and definitions:
    - Symptom: Observable psychological or behavioral symptoms (e.g., anxiety, insomnia).
    - Diagnosis: Mentioned or diagnosed mental health disorders (e.g., generalized anxiety disorder).
    - Treatment: Non-pharmacological therapy methods (e.g., cognitive behavioral therapy).
    - Drug: Names of medications or pharmaceutical substances (e.g., Xanax, Sertraline).
    - Dosage: Dosage instructions (e.g., twice a day, 50 mg).
    - Duration: Duration of medication use or symptom persistence (e.g., for one week, for 3 months).
    - Strength: Concentration of the medication (e.g., 500, 1.0).
    - Unit: Unit of measurement for strength (e.g., mg, ml, g).
    - Risk: Indicators of psychological or physical risk (e.g., suicidal thoughts, self-harm).

    ## Evaluation Criteria

    [The Start of Correctness Criterion]
    Does the model correctly extract the entity spans and assign the correct entity labels **based only on the question text**?

    - accurate: Most or all entities are correct, with minor or no issues.
    - mixed: Some correct extractions alongside boundary or label errors.
    - poor: Many incorrect, misaligned, mislabeled, or spurious entities.
    [The End of Correctness Criterion]

    [The Start of Completeness Criterion]
    How well did the model cover all important entities mentioned **in the question**?

    - full: All relevant entities present in the question are extracted.
    - partial: Some entities are extracted (about half or more).
    - low: Less than half of the important entities are captured.
    [The End of Completeness Criterion]

    ## Output Format
    Please return your evaluation in the following exact format as a markdown-style JSON block:

    {{
      "correctness": {{
        "verdict": "accurate/mixed/inaccurate",
        "reason": "Your detailed reasoning here..."
      }},
      "completeness": {{
        "verdict": "full/partial/missing",
        "reason": "Your detailed reasoning here..."
      }},
      "notes": "Optional comments if needed"
    }}

    """


## DeepSeek

In [ ]:
import os
os.environ["DEEPSEEK_API_KEY"] = "key"

In [ ]:
import requests

headers = {
    "Authorization": f"Bearer {os.getenv('DEEPSEEK_API_KEY')}",
    "Content-Type": "application/json"
}
payload = {
    "model": "deepseek-chat",
    "messages": [{"role": "user", "content": "مرحبا"}],
    "max_tokens": 10
}

try:
    r = requests.post("https://api.deepseek.com/chat/completions", headers=headers, json=payload, timeout=30)
    print(r.status_code)
    print(r.text)
except Exception as e:
    print("Error:", e)

In [ ]:
import re
import json
from openai import OpenAI

client = OpenAI(
    api_key=os.getenv("DEEPSEEK_API_KEY"),
    base_url="https://api.deepseek.com"
)

def score_with_deepseek(prompt):
    try:
        response = client.chat.completions.create(
            model="deepseek-chat",
            messages=[{"role": "user", "content": prompt}],
            max_tokens=1000,
            temperature=0,

            # Added for compatibility with the current DeepSeek API.
            # JSON mode forces valid JSON output and avoids parsing failures
            # caused by malformed or nested JSON responses.
            response_format={"type": "json_object"},
            extra_body={
                "thinking": {"type": "disabled"}
            }
        )

        content = response.choices[0].message.content.strip()

        if not content:
            raise ValueError("DeepSeek returned empty content")

        # JSON mode returns the full JSON object directly.
        return json.loads(content)

    except Exception as e:
        return {
            "correctness": {"verdict": "error", "reason": str(e)},
            "completeness": {"verdict": "error", "reason": str(e)},
            "notes": ""
        }

In [ ]:
# To test connection
test_prompt = build_prompt(df.iloc[0]['question'], df.iloc[0]['LLaMA Cleaned'])
print(score_with_deepseek(test_prompt))

{'correctness': {'verdict': 'accurate', 'reason': "The model correctly identified 'الخوف من عدم الإنجاب' (fear of not having children) as a Symptom, which aligns with the definition of observable psychological symptoms. No other entities from the schema are present in the question, so no mislabeling or spurious extractions occurred."}, 'completeness': {'verdict': 'full', 'reason': 'The only relevant entity in the question is the symptom of fear regarding future childlessness. The model extracted this completely, and no other entity types (Diagnosis, Treatment, Drug, etc.) are mentioned in the question.'}, 'notes': 'The extraction is accurate and complete given the limited entities present in the question.'}


In [ ]:
# Iterate and collect results
model_cols = ["LLaMA Cleaned", "ALLaM Cleaned", "GPT-4o Cleaned"]

In [ ]:
from collections import Counter

# Reset before each new run
summary = {col: {"correctness": Counter(), "completeness": Counter()} for col in model_cols}
results = {col: [] for col in model_cols}

In [ ]:
from tqdm import tqdm

for col in model_cols:
    for i, row in tqdm(df.iterrows(), total=len(df), desc=f"Scoring {col}"):
        prompt = build_prompt(row['question'], row[col])
        score = score_with_deepseek(prompt)

        results[col].append(score)

        # ✅ Update verdict counts
        if isinstance(score, dict):
            c = score.get("correctness", {}).get("verdict", "error")
            cm = score.get("completeness", {}).get("verdict", "error")
            summary[col]["correctness"][c] += 1
            summary[col]["completeness"][cm] += 1

In [ ]:
results_dict = {}

## LLaMA

In [ ]:
from tqdm import tqdm
import json

model_col = "LLaMA Cleaned"
judgment_field = "LLaMA Judgment"


for _, row in tqdm(df.iterrows(), total=len(df), desc=f"Evaluating {model_col}"):
    key = row["question"]

    # Initialize if new
    if key not in results_dict:
        results_dict[key] = {
            model_col: row[model_col]
        }

    # Skip if already evaluated
    if judgment_field in results_dict[key]:
        continue

    prompt = build_prompt(row["question"], row[model_col])
    score = score_with_deepseek(prompt)
    results_dict[key][judgment_field] = score

# Save after full column is done
with open("ner_judgment_results.json", "w", encoding="utf-8") as f:
    json.dump(results_dict, f, ensure_ascii=False, indent=2)


Evaluating LLaMA Cleaned: 100%|██████████| 499/499 [15:30<00:00,  1.86s/it]


In [ ]:
llama = pd.DataFrame(results_dict.values())

In [ ]:
llama.tail()

,LLaMA Cleaned,LLaMA Judgment
494,"[{""Symptom"": ""قلق، توتر، وسواس قهري، تأتأة في ...","{'correctness': {'verdict': 'mixed', 'reason':..."
495,"[{""Symptom"": ""الخوف، تنبؤات من كلام الأهل، بصم...","{'correctness': {'verdict': 'mixed', 'reason':..."
496,"[{""Symptom"": """", ""Diagnosis"": ""الذهانية"", ""Tre...","{'correctness': {'verdict': 'accurate', 'reaso..."
497,"[{""Symptom"": ""فقدان التركيز، النوم، الضغط، الغ...","{'correctness': {'verdict': 'mixed', 'reason':..."
498,"[{""Symptom"": """", ""Diagnosis"": """", ""Treatment"":...","{'correctness': {'verdict': 'poor', 'reason': ..."


In [ ]:
llama.insert(0, 'Question', df['question'])

In [ ]:
llama.head()

,Question,LLaMA Cleaned,LLaMA Judgment
0,هل يعتبر الخوف من عدم الإنجاب مستقبلاً حالة عا...,"[{""Symptom"": ""الخوف من عدم الإنجاب"", ""Diagnosi...","{'correctness': {'verdict': 'accurate', 'reaso..."
1,من سنه تقريبا و انا أذي نفسي ب اكثر من طريقة و...,"[{""Symptom"": ""أذية النفس، أفكار بإنهاء الحياة""...","{'correctness': {'verdict': 'mixed', 'reason':..."
2,السلام عليكم مشكلتي تقتصر على تكرار كلمة معينة...,"[{""Symptom"": ""تكرار كلمة معينة لمدة طويلة من ا...","{'correctness': {'verdict': 'mixed', 'reason':..."
3,اكتئاب وفوبيا من المجتمع وانعزال وانطوائية وتع...,"[{""Symptom"": ""اكتئاب، فوبيا، انعزال، انطوائية،...","{'correctness': {'verdict': 'mixed', 'reason':..."
4,عمري 37 سنة وحامل في الشهر الثامن . منذ الشهر ...,"[{""Symptom"": ""قلق شديد مزمن"", ""Diagnosis"": ""ال...","{'correctness': {'verdict': 'mixed', 'reason':..."


In [ ]:
llama.to_excel("zero_llama_judgments.xlsx", index=False)

from google.colab import files
files.download("zero_llama_judgments.xlsx")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

## Allam

In [ ]:
import json
from tqdm import tqdm

model_col = "ALLaM Cleaned"
judgment_field = "ALLaM Judgment"

# Load existing results
with open("ner_judgment_results.json", "r", encoding="utf-8") as f:
    results_dict = json.load(f)

for _, row in tqdm(df.iterrows(), total=len(df), desc=f"Re-evaluating {model_col}"):
    key = row["question"]

    # Initialize model output in results if missing
    if key not in results_dict:
        results_dict[key] = {}
    results_dict[key][model_col] = row[model_col]

    # Always re-run ALLaM's judgment
    prompt = build_prompt(row["question"], row[model_col])
    score = score_with_deepseek(prompt)
    results_dict[key][judgment_field] = score

# Save updated JSON
with open("ner_judgment_results.json", "w", encoding="utf-8") as f:
    json.dump(results_dict, f, ensure_ascii=False, indent=2)

Re-evaluating ALLaM Cleaned: 100%|██████████| 499/499 [16:04<00:00,  1.93s/it]


In [ ]:
from google.colab import files
files.download("ner_judgment_results.json")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
allam = pd.DataFrame(results_dict.values())

In [ ]:
allam.insert(0, 'Question', df['question'])

In [ ]:
allam.tail()

,Question,LLaMA Cleaned,LLaMA Judgment,ALLaM Cleaned,ALLaM Judgment
494,قلق وتوتر ووسواس قهري وتأتأة في الكلام وعجز دا...,"[{""Symptom"": ""قلق، توتر، وسواس قهري، تأتأة في ...","{'correctness': {'verdict': 'mixed', 'reason':...","[{""Symptom"": ""قلق"", ""Diagnosis"": ""وسواس قهري"",...","{'correctness': {'verdict': 'poor', 'reason': ..."
495,اعاني من الخوف وتنبؤات من كلام اهلي وبصمتهم ان...,"[{""Symptom"": ""الخوف، تنبؤات من كلام الأهل، بصم...","{'correctness': {'verdict': 'mixed', 'reason':...","[{""Diagnosis"": ""اضطراب القلق"", ""Risk"": ""الانتح...","{'correctness': {'verdict': 'mixed', 'reason':..."
496,لدي خمس سنوات علاج من الذهانية ولم اشفى تماما ...,"[{""Symptom"": """", ""Diagnosis"": ""الذهانية"", ""Tre...","{'correctness': {'verdict': 'accurate', 'reaso...","[{""Dosage"": ""..."", ""Duration"": ""لم اشف تماما ب...","{'correctness': {'verdict': 'mixed', 'reason':..."
497,أحس بفقدان التركيز كأنني أنسى معلومة أخبرتني ب...,"[{""Symptom"": ""فقدان التركيز، النوم، الضغط، الغ...","{'correctness': {'verdict': 'mixed', 'reason':...","[{""Symptom"": ""فقدان التركيز"", ""Drug"": ""..."", ""...","{'correctness': {'verdict': 'mixed', 'reason':..."
498,انا احس فيني الاكتئاب الجزئي او عسر المزاج نفس...,"[{""Symptom"": """", ""Diagnosis"": """", ""Treatment"":...","{'correctness': {'verdict': 'poor', 'reason': ...","[{""Diagnosis"": ""اكتئاب جزئي"", ""Treatment"": ""ال...","{'correctness': {'verdict': 'poor', 'reason': ..."


In [ ]:
# allam.to_excel("final_judgments.xlsx", index=False)

# from google.colab import files
# files.download("final_judgments.xlsx")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
allam.to_excel("zero_allam_judgments.xlsx", index=False)
files.download("zero_allam_judgments.xlsx")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
files.download("ner_judgment_results.json")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
# import pandas as pd


# # Loop through each question entry in results_dict
# for question, data in results_dict.items():
#     for model_col in ["LLaMA Cleaned", "ALLaM Cleaned", "GPT-4o Cleaned"]:
#         # Find the corresponding judgment field for that model
#         judgment_field = model_col.replace("Cleaned", "Judgment").strip()

#         if judgment_field in data:
#             score = data[judgment_field]
#             if isinstance(score, dict):
#                 flattened.append({
#                     "Question": question,
#                     "Model": model_col,
#                     "Correctness": score["correctness"]["verdict"],
#                     "Correctness Reason": score["correctness"]["reason"],
#                     "Completeness": score["completeness"]["verdict"],
#                     "Completeness Reason": score["completeness"]["reason"],
#                     "Notes": score.get("notes", "")
#                 })

# # Create a DataFrame and export
# df_results = pd.DataFrame(flattened)
# df_results.to_excel("NER_Evaluation_by_DeepSeek.xlsx", index=False)
# print(f"Exported {len(df_results)} rows to NER_Evaluation_by_DeepSeek.xlsx")

Exported 1497 rows to NER_Evaluation_by_DeepSeek.xlsx


In [ ]:
# df_results.head()

,Question,Model,Correctness,Correctness Reason,Completeness,Completeness Reason,Notes
0,هل يعتبر الخوف من عدم الإنجاب مستقبلاً حالة عا...,LLaMA Cleaned,accurate,The model correctly identified 'الخوف من عدم ا...,full,The question only mentions a psychological sym...,"The model performed well in this case, as the ..."
1,من سنه تقريبا و انا أذي نفسي ب اكثر من طريقة و...,LLaMA Cleaned,mixed,The model correctly identified 'أذي نفسي' as a...,partial,The model captured some key entities like 'Sym...,The model's performance is mixed due to some r...
2,السلام عليكم مشكلتي تقتصر على تكرار كلمة معينة...,LLaMA Cleaned,accurate,The model correctly identifies and labels the ...,full,The model captures all relevant entities menti...,The model performs well in both correctness an...
3,اكتئاب وفوبيا من المجتمع وانعزال وانطوائية وتع...,LLaMA Cleaned,accurate,The model correctly identified and labeled all...,full,The model extracted all the important entities...,The model performed well in both correctness a...
4,عمري 37 سنة وحامل في الشهر الثامن . منذ الشهر ...,LLaMA Cleaned,accurate,The model correctly extracted and labeled the ...,full,The model captured all relevant entities menti...,The model performed well in both correctness a...


In [ ]:
# df_results['Model'].unique()

array(['LLaMA Cleaned', 'ALLaM Cleaned'], dtype=object)

In [ ]:
# files.download("NER_Evaluation_by_DeepSeek.xlsx")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

## GPT-4o

In [ ]:
test_score = score_with_deepseek(
    build_prompt(
        df.iloc[0]["question"],
        df.iloc[0]["GPT-4o Cleaned"]
    )
)

print(test_score)

{'correctness': {'verdict': 'accurate', 'reason': "The model correctly identified the only clear symptom in the question: 'الخوف من عدم الإنجاب مستقبلاً' (fear of not having children in the future). This is a psychological symptom, and the label 'Symptom' is appropriate. No other entities from the schema are present in the question, so no mislabeling or spurious extractions occurred."}, 'completeness': {'verdict': 'full', 'reason': 'The question contains only one relevant entity according to the schema: the fear of future childlessness. The model extracted this entity, and no other entities (diagnosis, treatment, drug, etc.) are mentioned in the text. Therefore, all important entities are captured.'}, 'notes': "The model's extraction is correct and complete based on the provided schema and question text."}


In [ ]:
model_col = "GPT-4o Cleaned"
judgment_field = "GPT-4o Judgment"

for _, row in tqdm(df.iterrows(), total=len(df), desc=f"Evaluating {model_col}"):
    key = row["question"]

    # Initialize if new
    if key not in results_dict:
        results_dict[key] = {model_col: row[model_col]}
    else:
        results_dict[key][model_col] = row[model_col]

    # Skip if already evaluated
    if judgment_field in results_dict[key]:
        continue

    prompt = build_prompt(row["question"], row[model_col])
    score = score_with_deepseek(prompt)
    results_dict[key][judgment_field] = score

# Save
with open("ner_judgment_results.json", "w", encoding="utf-8") as f:
    json.dump(results_dict, f, ensure_ascii=False, indent=2)


Evaluating GPT-4o Cleaned: 100%|██████████| 499/499 [00:00<00:00, 20210.29it/s]


In [ ]:
import json
import time
from tqdm.auto import tqdm

model_col = "GPT-4o Cleaned"
judgment_field = "GPT-4o Judgment"
results_file = "ner_judgment_results.json"

# Load previously completed zero-shot judgments.
with open(results_file, "r", encoding="utf-8") as f:
    results_dict = json.load(f)

for _, row in tqdm(
    df.iterrows(),
    total=len(df),
    desc="Retrying zero-shot GPT-4o judgments"
):
    key = row["question"]

    if key not in results_dict:
        results_dict[key] = {}

    results_dict[key][model_col] = row[model_col]

    existing_score = results_dict[key].get(judgment_field, {})
    existing_verdict = (
        existing_score.get("correctness", {}).get("verdict")
        if isinstance(existing_score, dict)
        else None
    )

    # Preserve valid completed judgments.
    if existing_verdict and existing_verdict != "error":
        continue

    # Remove the previous failed judgment before retrying.
    results_dict[key].pop(judgment_field, None)

    prompt = build_prompt(row["question"], row[model_col])
    score = score_with_deepseek(prompt)

    # Do not save failed API calls as completed judgments.
    if score.get("correctness", {}).get("verdict") == "error":
        print(f"Judge error: {key[:60]}...")
    else:
        results_dict[key][judgment_field] = score

    # Save progress after every processed question.
    with open(results_file, "w", encoding="utf-8") as f:
        json.dump(results_dict, f, ensure_ascii=False, indent=2)

    time.sleep(0.2)

valid_count = sum(
    isinstance(record.get(judgment_field), dict)
    and record[judgment_field].get("correctness", {}).get("verdict") != "error"
    for record in results_dict.values()
)

print(f"Valid GPT-4o judgments: {valid_count} / {len(df)}")

Retrying zero-shot GPT-4o judgments:   0%|          | 0/499 [00:00<?, ?it/s]

Valid GPT-4o judgments: 499 / 499


In [ ]:
for field in ["GPT-4o Judgment", "ALLaM Judgment", "LLaMA Judgment"]:
    count = sum(
        isinstance(record.get(field), dict)
        and record[field].get("correctness", {}).get("verdict") != "error"
        for record in results_dict.values()
    )
    print(f"{field}: {count} / {len(df)}")

GPT-4o Judgment: 499 / 499
ALLaM Judgment: 499 / 499
LLaMA Judgment: 499 / 499


In [ ]:
import json
import time
from tqdm.auto import tqdm

results_file = "ner_judgment_results.json"

with open(results_file, "r", encoding="utf-8") as f:
    results_dict = json.load(f)


def retry_incomplete_judgments(model_col, judgment_field):
    pending_questions = []

    for question, data in results_dict.items():
        score = data.get(judgment_field, {})
        verdict = (
            score.get("correctness", {}).get("verdict")
            if isinstance(score, dict)
            else None
        )

        if verdict is None or verdict == "error":
            pending_questions.append(question)

    print(f"{model_col}: pending judgments = {len(pending_questions)}")

    for question in tqdm(
        pending_questions,
        desc=f"Retrying {model_col}"
    ):
        row = df[df["question"] == question].iloc[0]

        results_dict[question][model_col] = row[model_col]
        results_dict[question].pop(judgment_field, None)

        prompt = build_prompt(row["question"], row[model_col])
        score = score_with_deepseek(prompt)

        if score.get("correctness", {}).get("verdict") == "error":
            print(f"Judge error: {question[:60]}...")
        else:
            results_dict[question][judgment_field] = score

        with open(results_file, "w", encoding="utf-8") as f:
            json.dump(results_dict, f, ensure_ascii=False, indent=2)

        time.sleep(0.2)


retry_incomplete_judgments("ALLaM Cleaned", "ALLaM Judgment")
retry_incomplete_judgments("LLaMA Cleaned", "LLaMA Judgment")

ALLaM Cleaned: pending judgments = 1


Retrying ALLaM Cleaned:   0%|          | 0/1 [00:00<?, ?it/s]

LLaMA Cleaned: pending judgments = 1


Retrying LLaMA Cleaned:   0%|          | 0/1 [00:00<?, ?it/s]

In [ ]:
gpt = pd.DataFrame(results_dict.values())

In [ ]:
gpt.insert(0, 'Question', df['question'])

In [ ]:
gpt.head()

,Question,LLaMA Cleaned,LLaMA Judgment,ALLaM Cleaned,ALLaM Judgment,GPT-4o Cleaned,GPT-4o Judgment
0,هل يعتبر الخوف من عدم الإنجاب مستقبلاً حالة عا...,"[{""Symptom"": ""الخوف من عدم الإنجاب"", ""Diagnosi...","{'correctness': {'verdict': 'accurate', 'reaso...","[{""Symptom"": ""الخوف من عدم الإنجاب مستقبلاً"", ...","{'correctness': {'verdict': 'mixed', 'reason':...","[{""Symptom"": ""الخوف من عدم الإنجاب مستقبلاً"", ...","{'correctness': {'verdict': 'accurate', 'reaso..."
1,من سنه تقريبا و انا أذي نفسي ب اكثر من طريقة و...,"[{""Symptom"": ""أذية النفس، أفكار بإنهاء الحياة""...","{'correctness': {'verdict': 'mixed', 'reason':...","[{""symptom"": "","", ""diagnosis"": ""من سنه تقريبا ...","{'correctness': {'verdict': 'poor', 'reason': ...","[{""Symptom"": """", ""Diagnosis"": """", ""Treatment"":...","{'correctness': {'verdict': 'mixed', 'reason':..."
2,السلام عليكم مشكلتي تقتصر على تكرار كلمة معينة...,"[{""Symptom"": ""تكرار كلمة معينة لمدة طويلة من ا...","{'correctness': {'verdict': 'mixed', 'reason':...","[{""Symptom"": ""تكرار كلمة معينة لمدة طويلة"", ""R...","{'correctness': {'verdict': 'mixed', 'reason':...","[{""Symptom"": ""تكرار كلمة معينة"", ""Diagnosis"": ...","{'correctness': {'verdict': 'mixed', 'reason':..."
3,اكتئاب وفوبيا من المجتمع وانعزال وانطوائية وتع...,"[{""Symptom"": ""اكتئاب، فوبيا، انعزال، انطوائية،...","{'correctness': {'verdict': 'mixed', 'reason':...","[{""Symptom"": ""اكتئاب"", ""Diagnosis"": ""فوبيا من ...","{'correctness': {'verdict': 'poor', 'reason': ...","[{""Symptom"": ""فوبيا من المجتمع، انعزال، انطوائ...","{'correctness': {'verdict': 'mixed', 'reason':..."
4,عمري 37 سنة وحامل في الشهر الثامن . منذ الشهر ...,"[{""Symptom"": ""قلق شديد مزمن"", ""Diagnosis"": ""ال...","{'correctness': {'verdict': 'mixed', 'reason':...","[{""Symptom"": ""قلق شديد مزمن وخوف كبير"", \n""Dia...","{'correctness': {'verdict': 'mixed', 'reason':...","[{""Symptom"": ""قلق شديد مزمن"", ""Diagnosis"": ""ال...","{'correctness': {'verdict': 'mixed', 'reason':..."


In [ ]:
gpt.tail()

,Question,LLaMA Cleaned,LLaMA Judgment,ALLaM Cleaned,ALLaM Judgment,GPT-4o Cleaned,GPT-4o Judgment
494,قلق وتوتر ووسواس قهري وتأتأة في الكلام وعجز دا...,"[{""Symptom"": ""قلق، توتر، وسواس قهري، تأتأة في ...","{'correctness': {'verdict': 'mixed', 'reason':...","[{""Symptom"": ""قلق"", ""Diagnosis"": ""وسواس قهري"",...","{'correctness': {'verdict': 'poor', 'reason': ...","[{""Symptom"": ""قلق"", ""Diagnosis"": ""وسواس قهري"",...","{'correctness': {'verdict': 'mixed', 'reason':..."
495,اعاني من الخوف وتنبؤات من كلام اهلي وبصمتهم ان...,"[{""Symptom"": ""الخوف، تنبؤات من كلام الأهل، بصم...","{'correctness': {'verdict': 'mixed', 'reason':...","[{""Diagnosis"": ""اضطراب القلق"", ""Risk"": ""الانتح...","{'correctness': {'verdict': 'mixed', 'reason':...","[{""Symptom"": ""الخوف"", ""Diagnosis"": """", ""Treatm...","{'correctness': {'verdict': 'mixed', 'reason':..."
496,لدي خمس سنوات علاج من الذهانية ولم اشفى تماما ...,"[{""Symptom"": """", ""Diagnosis"": ""الذهانية"", ""Tre...","{'correctness': {'verdict': 'accurate', 'reaso...","[{""Dosage"": ""..."", ""Duration"": ""لم اشف تماما ب...","{'correctness': {'verdict': 'mixed', 'reason':...","[{""Symptom"": """", ""Diagnosis"": ""الذهانية"", ""Tre...","{'correctness': {'verdict': 'accurate', 'reaso..."
497,أحس بفقدان التركيز كأنني أنسى معلومة أخبرتني ب...,"[{""Symptom"": ""فقدان التركيز، النوم، الضغط، الغ...","{'correctness': {'verdict': 'mixed', 'reason':...","[{""Symptom"": ""فقدان التركيز"", ""Drug"": ""..."", ""...","{'correctness': {'verdict': 'mixed', 'reason':...","[{""Symptom"": ""فقدان التركيز"", ""Diagnosis"": """",...","{'correctness': {'verdict': 'accurate', 'reaso..."
498,انا احس فيني الاكتئاب الجزئي او عسر المزاج نفس...,"[{""Symptom"": """", ""Diagnosis"": """", ""Treatment"":...","{'correctness': {'verdict': 'poor', 'reason': ...","[{""Diagnosis"": ""اكتئاب جزئي"", ""Treatment"": ""ال...","{'correctness': {'verdict': 'poor', 'reason': ...","[{""Symptom"": ""ماعندي شهيه"", ""Diagnosis"": ""الاك...","{'correctness': {'verdict': 'mixed', 'reason':..."


In [ ]:
gpt.to_excel("zero_gpt_judgments.xlsx", index=False)

from google.colab import files
files.download("zero_gpt_judgments.xlsx")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
files.download("ner_judgment_results.json")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
import pandas as pd


# Loop through each question entry in results_dict
'''
for question, data in results_dict.items():
    for model_col in ["LLaMA Cleaned", "ALLaM Cleaned", "GPT-4o Cleaned"]:
        # Find the corresponding judgment field for that model
        judgment_field = model_col.replace("Cleaned", "Judgment").strip()

        if judgment_field in data:
            score = data[judgment_field]
            if isinstance(score, dict):
                flattened.append({
                    "Question": question,
                    "Model": model_col,
                    "Correctness": score["correctness"]["verdict"],
                    "Correctness Reason": score["correctness"]["reason"],
                    "Completeness": score["completeness"]["verdict"],
                    "Completeness Reason": score["completeness"]["reason"],
                    "Notes": score.get("notes", "")
                })

# Create a DataFrame and export
df_results = pd.DataFrame(flattened)
df_results.to_excel("NER_Evaluation_by_DeepSeek.xlsx", index=False)
print(f"Exported {len(df_results)} rows to NER_Evaluation_by_DeepSeek.xlsx")
'''

In [ ]:
#df_results.head()

,Question,Model,Correctness,Correctness Reason,Completeness,Completeness Reason,Notes
0,هل يعتبر الخوف من عدم الإنجاب مستقبلاً حالة عا...,LLaMA Cleaned,accurate,The model correctly identified 'الخوف من عدم ا...,full,The question only mentions a psychological sym...,"The model performed well in this case, as the ..."
1,من سنه تقريبا و انا أذي نفسي ب اكثر من طريقة و...,LLaMA Cleaned,mixed,The model correctly identified 'أذي نفسي' as a...,partial,The model captured some key entities like 'Sym...,The model's performance is mixed due to some r...
2,السلام عليكم مشكلتي تقتصر على تكرار كلمة معينة...,LLaMA Cleaned,accurate,The model correctly identifies and labels the ...,full,The model captures all relevant entities menti...,The model performs well in both correctness an...
3,اكتئاب وفوبيا من المجتمع وانعزال وانطوائية وتع...,LLaMA Cleaned,accurate,The model correctly identified and labeled all...,full,The model extracted all the important entities...,The model performed well in both correctness a...
4,عمري 37 سنة وحامل في الشهر الثامن . منذ الشهر ...,LLaMA Cleaned,accurate,The model correctly extracted and labeled the ...,full,The model captured all relevant entities menti...,The model performed well in both correctness a...


In [ ]:
#df_results['Model'].unique()

array(['LLaMA Cleaned', 'ALLaM Cleaned', 'GPT-4o Cleaned'], dtype=object)

In [ ]:
#files.download("NER_Evaluation_by_DeepSeek.xlsx")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
import json
import pandas as pd
from openpyxl import load_workbook
from openpyxl.styles import Font

# Save the completed zero-shot results under a clear final name.
json_output = "zero_shot_ner_judgment_results_retried.json"

with open(json_output, "w", encoding="utf-8") as f:
    json.dump(results_dict, f, ensure_ascii=False, indent=2)

# Convert the published-style JSON structure into a long evaluation table.
flattened = []

for question, data in results_dict.items():
    for model_col in [
        "LLaMA Cleaned",
        "ALLaM Cleaned",
        "GPT-4o Cleaned"
    ]:
        judgment_field = model_col.replace("Cleaned", "Judgment").strip()
        score = data.get(judgment_field)

        if isinstance(score, dict):
            flattened.append({
                "Question": question,
                "Model": model_col,
                "Correctness": score.get("correctness", {}).get("verdict", ""),
                "Correctness Reason": score.get("correctness", {}).get("reason", ""),
                "Completeness": score.get("completeness", {}).get("verdict", ""),
                "Completeness Reason": score.get("completeness", {}).get("reason", ""),
                "Notes": score.get("notes", "")
            })

df_results = pd.DataFrame(flattened)

excel_output = "zero_shot_NER_Evaluation_by_DeepSeek_retried.xlsx"
df_results.to_excel(excel_output, index=False)

# Apply basic formatting.
workbook = load_workbook(excel_output)
worksheet = workbook.active
worksheet.freeze_panes = "A2"
worksheet.auto_filter.ref = worksheet.dimensions

for row in worksheet.iter_rows():
    for cell in row:
        cell.font = Font(name="Arial", size=10)

for cell in worksheet[1]:
    cell.font = Font(name="Arial", size=10, bold=True)

workbook.save(excel_output)

print(f"Saved JSON: {json_output}")
print(f"Exported {len(df_results)} rows to {excel_output}")
print(df_results["Model"].value_counts())

Saved JSON: zero_shot_ner_judgment_results_retried.json
Exported 1497 rows to zero_shot_NER_Evaluation_by_DeepSeek_retried.xlsx
Model
LLaMA Cleaned     499
ALLaM Cleaned     499
GPT-4o Cleaned    499
Name: count, dtype: int64
